In [1]:
!pip install shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import shap

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

In [2]:
df_raw = pd.read_csv('museum-visitors.csv')

print('Shape original:', df_raw.shape)
print(df_raw.head())

FileNotFoundError: [Errno 2] No such file or directory: 'museum-visitors.csv'

In [ ]:
# Converter coluna de data
df_raw['Month'] = pd.to_datetime(df_raw['Month'])

df = df_raw.melt(
    id_vars=['Month'],
    var_name='Museu',
    value_name='Visitantes'
)

# Remover linhas sem dados
df = df.dropna(subset=['Visitantes'])

print('Shape após melt:', df.shape)
print(df.head(10))

In [ ]:
# Extrair features temporais da data
df['Mes'] = df['Month'].dt.month
df['Ano'] = df['Month'].dt.year
df['Trimestre'] = df['Month'].dt.quarter

# Flag de verão
df['Verao'] = df['Mes'].isin([6, 7, 8]).astype(int)

# Encodar o nome do museu para número
le = LabelEncoder()
df['Museu_enc'] = le.fit_transform(df['Museu'])

print('Features criadas:')
print(df.head(10))
print('\nMuseus e seus códigos:')
for i, museu in enumerate(le.classes_):
    print(f'  {i} = {museu}')

In [ ]:
# Definir features e variável alvo
X = df[['Mes', 'Ano', 'Trimestre', 'Verao', 'Museu_enc']]
y = df['Visitantes']

# Split treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')
print(f'Média de visitantes no treino: {y_train.mean():.0f}')
print(f'Média de visitantes no teste: {y_test.mean():.0f}')

In [ ]:
modelos = {
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

resultados = {}

for nome, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados[nome] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}

    print(f'\n--- {nome} ---')
    print(f'MAE:  {mae:.0f} visitantes')
    print(f'RMSE: {rmse:.0f} visitantes')
    print(f'R²:   {r2:.4f}')

df_resultados = pd.DataFrame(resultados).T
print('\nComparação:')
print(df_resultados.round(4))

In [ ]:
# Modelo escolhido
melhor_modelo = modelos['Random Forest']
melhor_nome = 'Random Forest'

y_pred_final = melhor_modelo.predict(X_test_scaled)

print(f'=== Avaliação Final — {melhor_nome} ===')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_final):.0f} visitantes')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_final)):.0f} visitantes')
print(f'R²:   {r2_score(y_test, y_pred_final):.4f}')

# Visualização — real vs previsto
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, y_pred_final, alpha=0.5, color='#3498db')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Visitantes Reais')
ax.set_ylabel('Visitantes Previstos')
ax.set_title('Real vs Previsto — Random Forest')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print('Calculando SHAP values...')
explainer = shap.TreeExplainer(melhor_modelo)
shap_values = explainer.shap_values(X_test_scaled)

print('SHAP calculado com sucesso!')
print(f'Shape dos SHAP values: {shap_values.shape}')

In [ ]:
feature_names = ['Mes', 'Ano', 'Trimestre', 'Verao', 'Museu_enc']

# Summary plot importância e direção
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled,
                  feature_names=feature_names,
                  show=False)
plt.title('SHAP — Importância das Features', fontsize=14)
plt.tight_layout()
plt.show()

# Bar plot importância média absoluta
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_test_scaled,
                  feature_names=feature_names,
                  plot_type='bar',
                  show=False)
plt.title('SHAP — Importância Média (|SHAP|)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Explicação individual primeiro caso do teste
idx = 0

print(f'Museu: {le.inverse_transform([int(X_test.iloc[idx]["Museu_enc"])])[0]}')
print(f'Mês: {int(X_test.iloc[idx]["Mes"])} | Ano: {int(X_test.iloc[idx]["Ano"])}')
print(f'Visitantes reais:   {int(y_test.iloc[idx])}')
print(f'Visitantes previstos: {int(melhor_modelo.predict(X_test_scaled[[idx]])[0])}')

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[idx],
    X_test_scaled[idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False
)
plt.title('SHAP Force Plot — Explicação Individual', fontsize=12)
plt.tight_layout()
plt.show()

Interpretação para o Negócio


Quais features mais influenciam a previsão?


Pelo SHAP, a feature mais importante foi o museu em si — ou seja, qual museu é importa mais do que o mês ou o ano. Faz sentido, porque cada museu tem um público próprio e tamanho diferente.


Em segundo lugar veio o ano, mostrando que a visitação muda com o tempo. Em terceiro o mês, capturando a sazonalidade — verão e férias tendem a ter mais visitantes.


Como o totem se beneficia?


O totem pode mostrar a previsão de visitantes para o próximo mês, ajudando o museu a planejar quantos funcionários escalar e quando fazer manutenções.

Que decisão pode ser tomada?

Reforçar equipe nos meses de alta prevista e aproveitar os meses de baixa para manutenção e eventos promocionais.

Avaliação Crítica

Limitações:

O dataset é pequeno e poucas informações — não temos clima, feriados nem eventos especiais, que claramente afetam visitação.


Possíveis melhorias:


Adicionar dados de clima e feriados melhoraria muito o modelo. Usar dados diários em vez de mensais também ajudaria.


Riscos:


O R² de 0.92 pode indicar que o modelo memorizou os dados em vez de generalizar. Os dados vão até 2021, então o comportamento pós-pandemia pode não estar bem representado.

In [ ]:
import joblib

# Salvar modelo, scaler e encoder
joblib.dump(melhor_modelo, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

print('Modelo salvo com sucesso!')